# 🧪 Lab 09: Find the Cause of Death

Welcome to the performance cause-of-death bay. By now we can find generated stages and identify their boundaries. This lab asks the production question: is WholeStageCodegen attacking the expensive part of the job?

**Mission Objective:** compare a CPU-heavy local-expression workload with a shuffle-heavy workload. We inspect the physical plans first, execute both, and read Spark UI stage metrics to connect plan shape with runtime, data movement, and spill.

**Deterministic Guardrail:** both workloads run on the same local Spark session with the same input scale and two partitions. This is a diagnostic comparison, not a universal performance ranking.


### Step 1: Define the session and metrics harvester
The Spark UI remains enabled so the notebook can query its local REST API after each action. The harvester collects completed-stage task time, shuffle read/write, and spill metrics.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import json
import time
from urllib.request import urlopen
from urllib.parse import quote, urlparse

spark = (SparkSession.builder
    .master("local[2]")
    .appName("lab-09-find-the-cause")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))

def ui_base_urls(sc):
    url = getattr(sc, "uiWebUrl", None)
    if callable(url):
        url = url()
    if not url:
        return []
    url = url.rstrip("/")
    parsed = urlparse(url)
    candidates = [url]
    if parsed.port:
        candidates.append(f"http://127.0.0.1:{parsed.port}")
    return list(dict.fromkeys(candidates))

def completed_stage_metrics(spark):
    app_id = quote(spark.sparkContext.applicationId, safe="/")
    endpoint_path = f"/api/v1/applications/{app_id}/stages?status=complete"
    for base in ui_base_urls(spark.sparkContext):
        try:
            with urlopen(base + endpoint_path, timeout=10) as response:
                stages = json.loads(response.read().decode("utf-8"))
            latest = {}
            for stage in stages:
                sid = int(stage.get("stageId", -1))
                attempt = int(stage.get("attemptId", 0))
                if sid not in latest or attempt > int(latest[sid].get("attemptId", 0)):
                    latest[sid] = stage
            return list(latest.values())
        except Exception:
            pass
    return []

print("Spark UI base URLs:", ui_base_urls(spark.sparkContext))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:26:34 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:26:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/24 06:26:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen enabled: true
Spark UI base URLs: ['http://10.255.255.254:4040', 'http://127.0.0.1:4040']


### Step 2: Build the two suspects
The CPU-heavy local-expression workload keeps data local and repeatedly evaluates native arithmetic. The shuffle-heavy workload repartitions by a key and aggregates after redistribution, making data movement a first-class part of the plan.


In [2]:
def cpu_bound_query():
    return (spark.range(0, 2_000_000, 1, numPartitions=2)
        .where((F.col("id") % 7) == 0)
        .select((
            F.sqrt(F.col("id") + 1.0)
            + F.log1p(F.col("id") + 1.0)
            + F.sin(F.col("id") * 0.001)
            + F.cos(F.col("id") * 0.002)
        ).alias("work")))

def shuffle_bound_query():
    return (spark.range(0, 2_000_000, 1, numPartitions=2)
        .select((F.col("id") % 100_000).alias("key"), F.col("id").alias("value"))
        .repartition(8, "key")
        .groupBy("key")
        .agg(F.sum("value").alias("total")))


### Step 3: Inspect plan shape before execution
The plan tells us where codegen and exchanges exist. It does not yet tell us how much time or data movement each stage consumed.


In [3]:
cpu_query = cpu_bound_query()
shuffle_query = shuffle_bound_query()

print("=== CPU-heavy local-expression physical plan ===")
cpu_query.explain("formatted")

print("=== Shuffle-heavy physical plan ===")
shuffle_query.explain("formatted")


=== CPU-heavy local-expression physical plan ===


== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((id#0L % 7) = 0)

(3) Project [codegen id : 1]
Output [1]: [(((SQRT((cast(id#0L as double) + 1.0)) + LOG1P((cast(id#0L as double) + 1.0))) + SIN((cast(id#0L as double) * 0.001))) + COS((cast(id#0L as double) * 0.002))) AS work#4]
Input [1]: [id#0L]


=== Shuffle-heavy physical plan ===
== Physical Plan ==
* HashAggregate (5)
+- * HashAggregate (4)
   +- Exchange (3)
      +- * Project (2)
         +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#5L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [2]: [(id#5L % 100000) AS key#6L, id#5L AS value#7L]
Input [1]: [id#5L]

(3) Exchange
Input [2]: [key#6L, value#7L]
Arguments: hashpartitioning(key#6L, 8), REPARTITION_BY_NUM, [plan_id=31]

(4) HashAggregate [cod

### Step 4: Execute and harvest stage metrics
We execute the already-planned JVM row pipeline with `query._jdf.queryExecution().toRdd().count()`. That forces all output rows to be produced without collecting hundreds of thousands of rows back to Python and without adding a new SQL aggregation that would introduce another `Exchange`.

We record elapsed wall time, completed-stage count, executor run time, shuffle read/write, and spill bytes.


In [4]:
def run_and_measure(name, query):
    before_ids = {int(s.get("stageId", -1)) for s in completed_stage_metrics(spark)}

    start = time.perf_counter()

    # Force the executed JVM plan without collecting its rows into Python.
    # Because this action happens on the already-planned RDD, it does not add
    # a new SQL aggregation or shuffle to the physical plan under investigation.
    row_count = query._jdf.queryExecution().toRdd().count()

    elapsed = time.perf_counter() - start

    # Give Spark's status store a moment to expose the completed-stage metrics.
    time.sleep(1)

    stages = completed_stage_metrics(spark)
    new_stages = [
        s for s in stages
        if int(s.get("stageId", -1)) not in before_ids
    ]

    totals = {
        "executorRunTime": sum(
            int(s.get("executorRunTime", 0) or 0) for s in new_stages
        ),
        "shuffleRead": sum(
            int(s.get("shuffleReadBytes", 0) or 0) for s in new_stages
        ),
        "shuffleWrite": sum(
            int(s.get("shuffleWriteBytes", 0) or 0) for s in new_stages
        ),
        "memoryBytesSpilled": sum(
            int(s.get("memoryBytesSpilled", 0) or 0) for s in new_stages
        ),
        "diskBytesSpilled": sum(
            int(s.get("diskBytesSpilled", 0) or 0) for s in new_stages
        ),
    }

    print(
        f"{name}: rows={row_count}, elapsed={elapsed:.3f}s, "
        f"stages={len(new_stages)}, metrics={totals}"
    )

    return totals


cpu_metrics = run_and_measure(
    "CPU-heavy local-expression",
    cpu_query,
)

shuffle_metrics = run_and_measure(
    "Shuffle-heavy",
    shuffle_query,
)


CPU-heavy local-expression: rows=285715, elapsed=1.333s, stages=1, metrics={'executorRunTime': 382, 'shuffleRead': 0, 'shuffleWrite': 0, 'memoryBytesSpilled': 0, 'diskBytesSpilled': 0}


Shuffle-heavy: rows=100000, elapsed=3.245s, stages=2, metrics={'executorRunTime': 4691, 'shuffleRead': 18301880, 'shuffleWrite': 18301880, 'memoryBytesSpilled': 0, 'diskBytesSpilled': 0}


### Step 5: Classify the dominant jurisdiction
The classification is deliberately qualitative. A local-expression workload with little shuffle points toward local computation and codegen as stronger suspects. Large shuffle read/write or spill points toward redistribution and memory pressure instead.


In [5]:
print(
    "CPU-heavy local-expression shuffle bytes:",
    cpu_metrics["shuffleRead"] + cpu_metrics["shuffleWrite"],
)
print(
    "Shuffle-heavy shuffle bytes:",
    shuffle_metrics["shuffleRead"] + shuffle_metrics["shuffleWrite"],
)
print(
    "CPU-heavy local-expression spill bytes:",
    cpu_metrics["memoryBytesSpilled"] + cpu_metrics["diskBytesSpilled"],
)
print(
    "Shuffle-heavy spill bytes:",
    shuffle_metrics["memoryBytesSpilled"] + shuffle_metrics["diskBytesSpilled"],
)

print(
    "Interpretation: codegen is most relevant when repeated local CPU work "
    "is a meaningful share of runtime; it cannot remove the cost of an Exchange."
)


CPU-heavy local-expression shuffle bytes: 0
Shuffle-heavy shuffle bytes: 36603760
CPU-heavy local-expression spill bytes: 0
Shuffle-heavy spill bytes: 0
Interpretation: codegen is most relevant when repeated local CPU work is a meaningful share of runtime; it cannot remove the cost of an Exchange.


# 📊 Post-Lab Analysis: Cause of Death Matters

This lab placed two different workloads under the same investigative tools. The physical plans showed where the codegen regions and exchanges were; the Spark UI metrics showed how much data movement and execution time each workload actually consumed.

### 1. Codegen Has a Jurisdiction

The CPU-heavy local-expression query is a natural target for WholeStageCodegen because its hot path repeatedly evaluates native expressions. The shuffle-heavy query contains an `Exchange`, so a meaningful part of its cost belongs to redistribution rather than the local Java loop.

### 2. A Plan Is Not a Cost Report

The physical plan tells us what Spark built. Stage metrics tell us how much executor time, shuffle, and spill appeared after execution. A star in the plan is not evidence that codegen dominated the job.

### 3. Amdahl Is Already in the Room

If only a small fraction of runtime is local CPU work, even a large improvement to that fraction produces a small end-to-end gain. Network, storage, skew, partitioning, and spill remain outside the reach of generated Java.

The useful question is not simply whether WholeStageCodegen is enabled. It is: **is codegen attacking the resource that currently owns the problem?**
